Excellent. Now we'll cover one of the most important concepts in Bagging that is frequently asked in interviews and exams.

# Part 3: Out-of-Bag (OOB) Evaluation

---

# What Problem Does OOB Solve?

Normally, after training a model, we evaluate it using:

* Train-Test Split
* Cross Validation

Example:

```text
Dataset
   │
   ├───────────┐
   │           │
Training     Testing
```

But Bagging has a clever trick.

Since every tree is trained on a different bootstrap sample, **some training samples are never seen by that tree.**

These unseen samples become a **free validation set**.

This is called the **Out-of-Bag (OOB) sample**.

---

# Let's Understand with an Example

Suppose your dataset has only 10 samples.

```text
1
2
3
4
5
6
7
8
9
10
```

Now Tree 1 creates a bootstrap sample.

Remember:

Bootstrap = Sampling **with replacement**

Tree 1 receives

```text
1
2
2
4
5
5
6
8
10
10
```

Notice something.

Missing samples are

```text
3
7
9
```

Tree 1 has **never seen**

* 3
* 7
* 9

during training.

Therefore,

```text
3
7
9
```

become Tree 1's OOB samples.

---

Now Tree 2

Bootstrap sample

```text
2
3
3
4
6
7
8
8
9
10
```

Missing

```text
1
5
```

These become Tree 2's OOB samples.

---

Tree 3

Bootstrap

```text
1
1
2
3
5
6
7
8
9
9
```

Missing

```text
4
10
```

Again,

those become OOB samples.

---

# Visual Representation

```text
Original Dataset

1
2
3
4
5
6
7
8
9
10

        │

──────────────────────────

Tree 1

Training

1
2
2
4
5
5
6
8
10
10

OOB

3
7
9

──────────────────────────

Tree 2

Training

2
3
3
4
6
7
8
8
9
10

OOB

1
5

──────────────────────────

Tree 3

Training

1
1
2
3
5
6
7
8
9
9

OOB

4
10
```

Every tree has different OOB samples.

---

# Why Is This Useful?

Imagine you have

```text
100 Trees
```

Every sample will be OOB for some of those trees.

Example

Sample

```text
Row 25
```

may be OOB for

```text
Tree 4

Tree 19

Tree 27

Tree 45

Tree 83
```

Only these trees are allowed to predict Row 25.

Then we use majority voting.

```text
Tree 4 → Spam

Tree 19 → Spam

Tree 27 → Ham

Tree 45 → Spam

Tree 83 → Spam

↓

Final

Spam
```

Now compare

```text
Prediction

Spam

Actual

Spam
```

Correct prediction.

We repeat this process for every training sample.

Finally,

```text
Correct Predictions

÷

Total Samples
```

gives

```text
OOB Score
```

---

# Why Approximately 36.8%?

This is one of the most asked interview questions.

Suppose

```text
1000 samples
```

Each bootstrap sample also contains

```text
1000 draws
```

Because sampling is **with replacement**, some rows are selected multiple times while others are not selected at all.

The probability that one specific row is **not selected in one draw** is:

```text
999 / 1000
```

After 1000 draws:

```text
(999/1000)^1000
```

For large datasets:

```text
≈ e^-1

≈ 0.368
```

Meaning

```text
36.8%
```

of the training samples are expected to be OOB for a given tree.

The remaining

```text
63.2%
```

are unique samples that the tree actually trains on.

---

# OOB vs Test Set

| Test Set              | OOB                                 |
| --------------------- | ----------------------------------- |
| Separate data         | Uses training data                  |
| Need train-test split | No extra split required             |
| Uses all models       | Only trees where the sample was OOB |
| Standard evaluation   | Built-in evaluation for Bagging     |

---

# Enabling OOB in Scikit-Learn

Very simple.

```python
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    oob_score=True,
    random_state=42
)

bag.fit(X_train, y_train)
```

Now,

```python
print(bag.oob_score_)
```

Example output

```text
0.9648
```

This means

```text
96.48%
```

OOB Accuracy.

---

# Important Condition

OOB works **only when**

```python
bootstrap=True
```

If

```python
bootstrap=False
```

there are no unused samples.

Hence,

```python
oob_score=True
```

will produce an error.

So,

```python
BaggingClassifier(
    bootstrap=True,
    oob_score=True
)
```

✅ Valid

```python
BaggingClassifier(
    bootstrap=False,
    oob_score=True
)
```

❌ Invalid

---

# OOB Prediction

Scikit-Learn also stores the OOB predictions.

```python
bag.oob_decision_function_
```

This returns the predicted probabilities for each training sample based **only on the trees where that sample was OOB**.

Example

```python
print(bag.oob_decision_function_[:5])
```

Output (example)

```text
[[0.98 0.02]
 [0.15 0.85]
 [0.91 0.09]
 [0.40 0.60]
 [0.05 0.95]]
```

Each row represents the class probabilities for one training sample.

For binary classification:

* First column → Probability of class 0
* Second column → Probability of class 1

---

# Advantages of OOB

* No need for a separate validation dataset.
* Makes full use of the available training data.
* Gives a good estimate of the model's generalization performance.
* Saves computation compared to k-fold cross-validation.
* Particularly useful when the dataset is small.

---

# Limitations of OOB

* Only available for bootstrap-based methods.
* Slightly less reliable than a well-designed cross-validation on some datasets.
* Not available when `bootstrap=False`.
* With very few estimators, the OOB estimate can be noisy because each sample is evaluated by only a small number of trees.

---

# OOB vs Cross Validation

| Feature                              | OOB   | Cross Validation           |
| ------------------------------------ | ----- | -------------------------- |
| Extra training required              | ❌ No  | ✅ Yes                      |
| Works only for Bagging/Random Forest | ✅ Yes | ❌ No (works for any model) |
| Uses bootstrap samples               | ✅ Yes | ❌ No                       |
| Computational cost                   | Low   | Higher                     |
| Accuracy estimate                    | Good  | Usually more robust        |

---

# Interview Questions

### Q1. Why is OOB possible only in Bagging?

**Answer:** Because Bagging uses bootstrap sampling (sampling with replacement), which leaves some training samples out of each bootstrap dataset. Those unseen samples become Out-of-Bag samples for that estimator.

---

### Q2. What is the expected percentage of OOB samples?

**Answer:** Approximately **36.8%** of the original training samples are OOB for any given tree, while about **63.2%** are unique samples included in its bootstrap training set.

---

### Q3. Should you always use OOB instead of a test set?

**Answer:** No. OOB is an excellent internal estimate during model development, but for the final evaluation of a model, it's still best practice to use an independent test set that the model has never influenced in any way.

---

## Next Topic

The next part will cover **BaggingRegressor**, where you'll learn how Bagging works for regression problems, followed by a comparison of **Bagging vs Random Forest**, which is the most important transition before studying Random Forest in depth.
